# Chatbot RAG — The Acts of the Apostles

## 1. Indexação dos embeddings com FAISS

Etapas:
1. Ler o `.txt` e separar o livro por capítulo (cada capítulo começa com `\fChap. N - Título`).
2. Quebrar cada capítulo em *chunks* menores.
3. Gerar os embeddings com `intfloat/multilingual-e5-small`, que é multilíngue: perguntas em português encontram trechos em inglês.
4. Indexar no FAISS e salvar o índice em disco.

In [46]:
import re
from pathlib import Path

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

SOURCE_FILE = Path("rag-file/The-Acts-of-the-Apostles.txt")
INDEX_DIR = Path("faiss-index")
EMBEDDING_MODEL = "intfloat/multilingual-e5-small"

### 1.1 Carregar o texto e separar por capítulo

No corpo do livro, cada capítulo começa com um *form feed* (`\f`) seguido de `Chap. N - Título`. O título pode ocupar duas linhas.
Tudo o que vem antes do capítulo 1 (licença, prefácio, sumário) fica de fora.
Cada quebra de página traz o número da página numa linha própria, seguido de `\f`. Esses números são removidos para não aparecerem no meio das frases.
O texto também tem quebras de linha no meio das frases, então os espaços em branco são normalizados.

In [47]:
text = SOURCE_FILE.read_text(encoding="utf-8")

# quebra da pagina
PAGE_BREAK_RE = re.compile(r"\n\s*\d{1,3}\s*(?:\f|$)")
# captura o cabecalho de cada capitulo e captura o numero e o tiulo
CHAPTER_RE = re.compile(r"\fChap\.\s*(\d+)\s*-\s*([^\n]+(?:\n[^\n]+)?)\n\n")
# lista dos 58 cabecalhos
matches = list(CHAPTER_RE.finditer(text))

chapters = []
for i, m in enumerate(matches):
    # index de onde o capitulo termina -> inicio do proximo cabecalho
    end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
    # recorte do final do cabecalho ate o inicio do outro
    body = PAGE_BREAK_RE.sub(" ", text[m.end():end])
    # limpa as quebras de pagina do body
    body = re.sub(r"\s+", " ", body).strip()
    chapters.append(Document(
        page_content=body,
        metadata={
            "chapter": int(m.group(1)), # pega o (\d+) do regex do chapter
            "title": re.sub(r"\s+", " ", m.group(2)).strip(), # pega o ([^\n]+(?:\n[^\n]+)?) do regear do chapter e limpa
            "source": SOURCE_FILE.name,
        },
    ))

print(f"{len(chapters)} capítulos")
chapters[0].metadata, chapters[0].page_content[:300]

58 capítulos


({'chapter': 1,
  'title': 'God’s Purpose for His Church',
  'source': 'The-Acts-of-the-Apostles.txt'},
 'The church is God’s appointed agency for the salvation of men. It was organized for service, and its mission is to carry the gospel to the world. From the beginning it has been God’s plan that through His church shall be reflected to the world His fullness and His sufficiency. The members of the chu')

### 1.2 Dividir em chunks

O `multilingual-e5-small` trunca a entrada em 512 *tokens*, ou seja, uns 2.000 caracteres em inglês.
Chunks de ~800 caracteres com 100 de sobreposição ficam bem abaixo desse limite.
Cada chunk herda o capítulo e o título como metadados.

In [48]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=[". ", "? ", "! ", "; ", ", ", " ", ""],
    keep_separator="end",
)
chunks = splitter.split_documents(chapters)

print(f"{len(chunks)} chunks")
chunks[0]

1317 chunks


Document(metadata={'chapter': 1, 'title': 'God’s Purpose for His Church', 'source': 'The-Acts-of-the-Apostles.txt'}, page_content='The church is God’s appointed agency for the salvation of men. It was organized for service, and its mission is to carry the gospel to the world. From the beginning it has been God’s plan that through His church shall be reflected to the world His fullness and His sufficiency. The members of the church, those whom He has called out of darkness into His marvelous light, are to show forth His glory. The church is the repository of the riches of the grace of Christ; and through the church will eventually be made manifest, even to “the principalities and powers in heavenly places,” the final and full display of the love of God. Ephesians 3:10. Many and wonderful are the promises recorded in the Scriptures regarding the church.')

### 1.3 Gerar os embeddings e indexar no FAISS

Os modelos E5 foram treinados com prefixos: `"passage: "` nos trechos indexados e `"query: "` nas perguntas. Sem eles, a qualidade da busca cai.
O `prompt` do sentence-transformers acrescenta o prefixo automaticamente: `encode_kwargs` vale para os documentos e `query_encode_kwargs` para as perguntas.

Com `normalize_embeddings=True`, ordenar pela distância L2 dá o mesmo resultado que ordenar pela similaridade de cosseno.

In [49]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={"normalize_embeddings": True, "prompt": "passage: "},
    query_encode_kwargs={"normalize_embeddings": True, "prompt": "query: "},
)

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local(str(INDEX_DIR))

print(f"{vectorstore.index.ntotal} vetores de dimensão {vectorstore.index.d} salvos em {INDEX_DIR}/")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19846.08it/s]


1317 vetores de dimensão 384 salvos em faiss-index/


### 1.4 Testar o retrieval

Depois de salvo, o índice pode ser recarregado sem gerar os embeddings de novo.
`allow_dangerous_deserialization` é necessário porque o docstore é salvo em pickle; só use com índices gerados por você.

In [50]:
vectorstore = FAISS.load_local(
    str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True
)

for query in ["What happened with Stephen?", "O que aconteceu com Estêvão?"]:
    print(f"=== {query}")
    for doc, score in vectorstore.similarity_search_with_score(query, k=4):
        print(f"[{score:.3f}] Cap. {doc.metadata['chapter']} - {doc.metadata['title']}")
        print(doc.page_content[:250], "...\n")

=== What happened with Stephen?
[0.299] Cap. 18 - Preaching Among the Heathen
Soon the murderous designs of the enemies of the gospel were carried out. Yielding to the influence of evil, the Lystrians became possessed with a satanic fury and, seizing Paul, mercilessly stoned him. The apostle thought that his end had come. The  ...

[0.311] Cap. 10 - The First Christian Martyr
Stephen, the foremost of the seven deacons, was a man of deep piety and broad faith. Though a Jew by birth, he spoke the Greek language and was familiar with the customs and manners of the Greeks. He therefore found opportunity to preach the gospel i ...

[0.311] Cap. 10 - The First Christian Martyr
Stopping their ears, that they might not hear his words, and uttering loud cries, they ran furiously upon him with one accord “and cast him out of the city.” “And they stoned Stephen, calling upon God, and saying, Lord Jesus, receive my spirit. And h ...

[0.324] Cap. 10 - The First Christian Martyr
He connected Jesus 

## 2. Gerar a resposta com o `flan-t5-base`

O `google/flan-t5-base` é um modelo *seq2seq* (encoder-decoder) do Google, com ~250 milhões de parâmetros, treinado para seguir instruções curtas.
Ele roda localmente, sem API, mas tem limites:
- entende bem só **inglês**;
- foi treinado com entradas de até **512 tokens**. Os 4 chunks somam ~750 tokens; o T5 aceita entradas maiores porque usa posições relativas, mas a qualidade pode cair;
- dá respostas **curtas e extrativas**, geralmente copiando um pedaço do contexto.

### 2.1 Carregar o modelo

In [74]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

GENERATOR_MODEL = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL)
generator = AutoModelForSeq2SeqLM.from_pretrained(GENERATOR_MODEL)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 2474.18it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


### 2.2 Recuperar os 4 trechos mais parecidos e responder

`similarity_search_with_score(question, k=4)` devolve os 4 chunks de **menor score**. Os textos são juntados num único contexto, e o modelo é instruído a responder em duas ou três frases.

Dois detalhes do prompt:
- **A pergunta vem antes do contexto.** Se a entrada passar do `max_length`, o tokenizer corta o final; assim se perde o fim do último trecho, e não a pergunta.
- **`max_length=1024`** para caber os 4 trechos (~750 tokens) sem truncar.

Na geração, `num_beams=4` explora 4 sequências candidatas em paralelo e fica com a mais provável, e `no_repeat_ngram_size=3` impede que o modelo repita a mesma sequência de 3 tokens.

In [ ]:
def answer(question, k=4):
    # k=4: os 4 trechos de menor score, ou seja, os mais parecidos com a pergunta
    results = vectorstore.similarity_search_with_score(question, k=k)
    context = "\n\n".join(doc.page_content for doc, _ in results)

    # a pergunta vem primeiro: se o prompt for truncado, perde-se o fim do contexto, não a pergunta
    prompt = (
        f"Question: {question}\n\n"
        "Answer the question in two or three sentences, using only the context below.\n\n"
        f"Context: {context}"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    output = generator.generate(
        **inputs, max_new_tokens=120, num_beams=4, no_repeat_ngram_size=3
    )

    return tokenizer.decode(output[0], skip_special_tokens=True), results

### 2.3 Testar

In [ ]:
for question in [
    "What happened with Stephen?",
    "How many souls were added to the church at Pentecost?",
    "Who freed Peter from prison?",
    "What did Paul do in Athens?",
    "O que aconteceu com Estêvão?",
]:
    response, results = answer(question)
    print(f"Pergunta: {question}")
    print(f"Resposta: {response}")
    print("Fontes:")
    for doc, score in results:
        print(f"  [{score:.3f}] Cap. {doc.metadata['chapter']} - {doc.metadata['title']}")
    print()